# Emotion Recognition — FER2013
Custom CNN trained from scratch on native 48x48 grayscale faces.
Transfer learning (MobileNet/EfficientNet) underperforms on FER2013 because ImageNet features do not transfer to tiny grayscale faces. A purpose-built CNN reliably reaches ~66-68%.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as     plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Conv2D, BatchNormalization, Activation,
                                     MaxPooling2D, Dropout, Flatten, Dense)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print(f'TF version: {tf.__version__}')
print(f'GPUs: {tf.config.list_physical_devices("GPU")}')

In [ ]:
TRAIN_DIR = '/kaggle/input/datasets/msambare/fer2013/train'
TEST_DIR  = '/kaggle/input/datasets/msambare/fer2013/test'
IMG_SIZE  = (48, 48)
BATCH_SIZE = 64
NUM_CLASSES = 7
EMOTIONS = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']

In [ ]:
counts = {}
for emotion in sorted(os.listdir(TRAIN_DIR)):
    counts[emotion] = len(os.listdir(os.path.join(TRAIN_DIR, emotion)))

plt.figure(figsize=(10, 4))
plt.bar(counts.keys(), counts.values(), color='steelblue')
plt.title('Training Set Class Distribution')
plt.xlabel('Emotion')
plt.ylabel('Number of Images')
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150)
plt.show()
print(counts)

In [ ]:
fig, axes = plt.subplots(2, 7, figsize=(14, 4))
for i, emotion in enumerate(sorted(os.listdir(TRAIN_DIR))):
    emotion_dir = os.path.join(TRAIN_DIR, emotion)
    files = os.listdir(emotion_dir)
    for row in range(2):
        img = plt.imread(os.path.join(emotion_dir, files[row]))
        axes[row, i].imshow(img, cmap='gray')
        if row == 0:
            axes[row, i].set_title(emotion, fontsize=8)
        axes[row, i].axis('off')
plt.suptitle('Sample Images per Emotion')
plt.tight_layout()
plt.savefig('sample_images.png', dpi=150)
plt.show()

## Data generators
Native 48x48 grayscale. Rescale to [0,1]. Augmentation on train only. No class weights (they hurt accuracy on FER2013).

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,
    rotation_range=12,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1,
    validation_split=0.1
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, color_mode='grayscale',
    batch_size=BATCH_SIZE, class_mode='categorical',
    subset='training', shuffle=True, seed=42
)

val_gen = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, color_mode='grayscale',
    batch_size=BATCH_SIZE, class_mode='categorical',
    subset='validation', shuffle=False, seed=42
)

test_gen = test_datagen.flow_from_directory(
    TEST_DIR, target_size=IMG_SIZE, color_mode='grayscale',
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)

print(f'Train: {train_gen.samples}, Val: {val_gen.samples}, Test: {test_gen.samples}')
print(f'Class indices: {train_gen.class_indices}')

## Model — VGG-style CNN from scratch
4 conv blocks (64,128,256,512) with BatchNorm + Dropout, then dense head. Proven architecture for FER2013.

In [ ]:
def conv_block(model, filters):
    for _ in range(2):
        model.add(Conv2D(filters, (3, 3), padding='same'))
        model.add(BatchNormalization())
        model.add(Activation('relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Dropout(0.25))

def build_model():
    model = Sequential()
    model.add(tf.keras.Input(shape=(48, 48, 1)))
    conv_block(model, 64)
    conv_block(model, 128)
    conv_block(model, 256)
    conv_block(model, 512)
    model.add(Flatten())
    model.add(Dense(512))
    model.add(BatchNormalization())
    model.add(Activation('relu'))
    model.add(Dropout(0.5))
    model.add(Dense(256))
    model.add(BatchNormalization())
    model.add(Activation('relu'))
    model.add(Dropout(0.5))
    model.add(Dense(NUM_CLASSES, activation='softmax'))
    return model

model = build_model()
model.summary()
print(f'Total params: {model.count_params():,}')

## Training
Single phase from scratch. Adam, lr=1e-3 with ReduceLROnPlateau. Up to 60 epochs, EarlyStopping patience=10.

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6, verbose=1),
    ModelCheckpoint('emotion_model.keras', monitor='val_accuracy', save_best_only=True, verbose=1)
]

history = model.fit(
    train_gen,
    epochs=60,
    validation_data=val_gen,
    callbacks=callbacks,
    verbose=1
)

print(f'Best val_accuracy: {max(history.history["val_accuracy"]):.4f}')

In [ ]:
def plot_metric(h, metric):
    plt.figure(figsize=(10, 4))
    plt.plot(h.history[metric], label=f'Train {metric}')
    plt.plot(h.history[f'val_{metric}'], label=f'Val {metric}')
    plt.legend()
    plt.title(f'{metric.capitalize()} over epochs')
    plt.xlabel('Epoch')
    plt.tight_layout()
    plt.savefig(f'{metric}_curve.png', dpi=150)
    plt.show()

plot_metric(history, 'accuracy')
plot_metric(history, 'loss')

In [ ]:
model.load_weights('emotion_model.keras')
test_gen.reset()
test_loss, test_acc = model.evaluate(test_gen, verbose=1)
print(f'\nTest accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)')
print(f'Test loss:     {test_loss:.4f}')

In [ ]:
test_gen.reset()
y_pred_probs = model.predict(test_gen, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = test_gen.classes
class_names = list(test_gen.class_indices.keys())

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix — Test Set')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

print('\n' + classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
saved_model = tf.keras.models.load_model('emotion_model.keras')

converter = tf.lite.TFLiteConverter.from_keras_model(saved_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open('emotion_model.tflite', 'wb') as f:
    f.write(tflite_model)

size_mb = len(tflite_model) / 1024 / 1024
print(f'TFLite model size: {size_mb:.2f} MB')

In [ ]:
interpreter = tf.lite.Interpreter(model_content=tflite_model)
interpreter.allocate_tensors()
inp = interpreter.get_input_details()
out = interpreter.get_output_details()
print(f'Input:  shape={inp[0]["shape"]}, dtype={inp[0]["dtype"]}')
print(f'Output: shape={out[0]["shape"]}, dtype={out[0]["dtype"]}')

dummy = np.random.rand(1, 48, 48, 1).astype(np.float32)
interpreter.set_tensor(inp[0]['index'], dummy)
interpreter.invoke()
result = interpreter.get_tensor(out[0]['index'])[0]
print(f'Output sum: {result.sum():.4f} (should be ~1.0)')
print(f'Predicted class: {EMOTIONS[np.argmax(result)]}')

In [ ]:
from IPython.display import FileLink, display

files_to_download = [
    'emotion_model.tflite',
    'emotion_model.keras',
    'class_distribution.png',
    'sample_images.png',
    'accuracy_curve.png',
    'loss_curve.png',
    'confusion_matrix.png'
]

for fname in files_to_download:
    if os.path.exists(fname):
        display(FileLink(fname))
        print(f'✓ {fname}')
    else:
        print(f'✗ {fname} NOT FOUND')